# OOVI — LoRA for `gemma3:270m` (Colab)

This notebook trains one small LoRA adapter for the **narrow OOVI layers** on top of
[`google/gemma-3-270m-it`](https://huggingface.co/google/gemma-3-270m-it), merges it,
converts it to **GGUF Q8_0**, and writes a Modelfile that uses **the exact chat template
Ollama serves** for `gemma3:270m`.

Three things are deliberately *not* reinvented here:

1. **The prompts are the production prompts.** They are read verbatim from
   `data/template-copy.train.jsonl`, which `scripts/make-dataset.mjs` built with the same
   `src/ai/*Prompt.js` builders the app calls. Nothing is rephrased in Colab.
2. **The chat format is Ollama's.** HF's Gemma 3 template renders
   `<bos><start_of_turn>user\n{system}\n\n{user}<end_of_turn>\n<start_of_turn>model\n`;
   the notebook asserts that against the Go template captured from
   `ollama show gemma3:270m --modelfile`, and that template is embedded in the Modelfile.
3. **Loss is completion-only.** Prompt tokens are labelled `-100`; only the target JSON
   and the closing `<end_of_turn>` are trained.

The adapter is task-selectable (`choice`, `tune`, `copy`). Train all three together, or
run the notebook once per layer and point `OLLAMA_MODEL_TUNE` / `OLLAMA_MODEL_COPY` at the
separate imports.

**No training is run from the repo, and this notebook does not run in the repo's test
suite.** It is the Colab half; `scripts/tiny-eval.mjs --model <tag> --strict` is the local
half that measures the result.

## 1. Runtime, Drive, and Hugging Face access

In [ ]:
# Colab: Runtime -> Change runtime type -> T4 GPU (free tier is enough for 270m).
!nvidia-smi || echo "No GPU: switch the runtime to T4 before training."

from pathlib import Path
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive/oovi")
except Exception:
    DRIVE = Path("/content/oovi")
DRIVE.mkdir(parents=True, exist_ok=True)
print("output root:", DRIVE)

The HF repo is **gated**: accept the Gemma license on the model page, create a read token,
and put it in Colab Secrets as `HF_TOKEN` (key icon in the left sidebar).

In [ ]:
# Accept the Gemma license at https://huggingface.co/google/gemma-3-270m-it first.
try:
    from google.colab import userdata
    from huggingface_hub import login
    login(userdata.get("HF_TOKEN"))
except Exception as exc:
    print("HF login skipped:", exc)

In [ ]:
!pip install -q -U "transformers>=4.50,<5" "peft>=0.14" "datasets>=3" "accelerate>=1.0" sentencepiece protobuf gguf

## 2. Configuration

In [ ]:
from pathlib import Path
import json, statistics, torch

# --- what to train ---------------------------------------------------------
TASKS = ("choice", "tune", "copy")     # e.g. ("tune",) for one layer only
BASE_MODEL = "google/gemma-3-270m-it"

# --- where the dataset lives ----------------------------------------------
# Upload data/template-copy.train.jsonl + data/template-copy.eval.jsonl to
# Colab ("data" folder) or copy them into DRIVE/data.
CANDIDATES = [DRIVE / "data", Path("/content/data"), Path("data")]
DATA_DIR = next((p for p in CANDIDATES if (p / "template-copy.train.jsonl").exists()), None)
assert DATA_DIR is not None, "Upload template-copy.train.jsonl to /content/data or Drive/oovi/data"

# --- training --------------------------------------------------------------
MAX_SEQ_LEN = 2048
EPOCHS = 3
LR = 2e-4
BATCH = 4
GRAD_ACCUM = 4

RUN = "all" if set(TASKS) == {"choice", "tune", "copy"} else "-".join(TASKS)
MODEL_NAME = "oovi-gemma3-270m" if RUN == "all" else f"oovi-gemma3-270m-{RUN}"
OUT = DRIVE / RUN
OUT.mkdir(parents=True, exist_ok=True)
print("task(s):", TASKS, "| model tag:", MODEL_NAME + ":q8")
print("dataset:", DATA_DIR, "| output:", OUT)

## 3. Load the production prompts

In [ ]:
def split_prompt(full: str) -> tuple[str, str]:
    """Byte-for-byte copy of the split in src/ai/aiClient.js."""
    lines = str(full or "").split("\n")
    at = next((i for i, line in enumerate(lines) if line.strip() == "[User]"), -1)
    if at == -1:
        return "", str(full or "")
    system = "\n".join(line for line in lines[:at] if line.strip() != "[System]").strip()
    user = "\n".join(lines[at + 1:]).strip()
    return system, user


def read_jsonl(path):
    with open(path, encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]


train_all = read_jsonl(DATA_DIR / "template-copy.train.jsonl")
eval_all = read_jsonl(DATA_DIR / "template-copy.eval.jsonl")
train_records = [r for r in train_all if r["task"] in TASKS]
eval_records = [r for r in eval_all if r["task"] in TASKS]
assert train_records, f"no records for {TASKS}"

print("train:", len(train_records), "eval:", len(eval_records))
for task in sorted({r["task"] for r in train_records}):
    print(f"  {task}: {sum(r['task'] == task for r in train_records)}")

sample = train_records[0]
system, user = split_prompt(sample["prompt"])
print("\n--- system prefix (first 200 chars) ---\n", system[:200])
print("\n--- user brief (first 400 chars) ---\n", user[:400])
print("\n--- target ---\n", json.dumps(sample["target"], ensure_ascii=False)[:400])

## 4. Tokenizer and chat-template parity

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("chat template:\n", tokenizer.chat_template)
print("bos:", tokenizer.bos_token, tokenizer.bos_token_id,
      "| eos:", tokenizer.eos_token, tokenizer.eos_token_id,
      "| pad:", tokenizer.pad_token, tokenizer.pad_token_id)

# The tags must each be one token, or the mask boundary would be wrong.
for tag in ("<bos>", "<start_of_turn>", "<end_of_turn>"):
    ids = tokenizer(tag, add_special_tokens=False)["input_ids"]
    print(tag, ids)
    assert len(ids) == 1, f"{tag} is not a single token in this tokenizer"

In [ ]:
def ollama_render(system: str, user: str) -> str:
    """What the Modelfile TEMPLATE below renders for [system, user] + generation prompt."""
    return f"<start_of_turn>user\n{system}\n\n{user}<end_of_turn>\n<start_of_turn>model\n"


def render_messages(system: str, user: str) -> str:
    """The exact token sequence Ollama serves: BOS + the Go template's output."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": user})
    rendered = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    expected = tokenizer.bos_token + ollama_render(system, user)
    assert rendered == expected, (
        "HF chat template and the Ollama template disagree.\n"
        f"HF:     {rendered!r}\nOllama: {expected!r}"
    )
    return rendered


parity = render_messages(system, user)
print(repr(parity[:160]), "...")
print("\nparity OK: HF template + BOS == Ollama Modelfile template")

**Completion-only loss.** The prompt is tokenized once; the target JSON plus
`<end_of_turn>` is tokenized after it. Every prompt token gets label `-100`, so the
adapter only learns to produce the answer. The prompt used for training is the same
string that `ollama` will see, including BOS and the generation prompt.

In [ ]:
END = "<end_of_turn>"

def encode_record(record):
    system, user = split_prompt(record["prompt"])
    prompt_text = render_messages(system, user)
    target_text = json.dumps(record["target"], ensure_ascii=False, separators=(",", ":"))
    completion = target_text + END

    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    full_ids = tokenizer(prompt_text + completion, add_special_tokens=False)["input_ids"]
    assert full_ids[:len(prompt_ids)] == prompt_ids, "prompt is not a prefix of the full sequence"
    assert len(full_ids) > len(prompt_ids), "empty completion"

    labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]
    return {
        "input_ids": full_ids,
        "labels": labels,
        "prompt_tokens": len(prompt_ids),
        "target_tokens": len(full_ids) - len(prompt_ids),
    }


encoded, skipped = [], []
for record in train_records:
    item = encode_record(record)
    if len(item["input_ids"]) > MAX_SEQ_LEN:
        skipped.append((record["task"], len(item["input_ids"])))
        continue
    encoded.append(item)

assert encoded, "nothing fits MAX_SEQ_LEN"
print("encoded:", len(encoded), "| skipped (too long):", len(skipped))
if skipped:
    print("skipped examples:", skipped[:10])

lengths = [len(x["input_ids"]) for x in encoded]
prompt_lengths = [x["prompt_tokens"] for x in encoded]
target_lengths = [x["target_tokens"] for x in encoded]
print(f"sequence tokens: min={min(lengths)} median={statistics.median(lengths):.0f} max={max(lengths)}")
print(f"prompt tokens:   median={statistics.median(prompt_lengths):.0f}")
print(f"target tokens:   median={statistics.median(target_lengths):.0f}")
print(f"completion share of loss: {sum(target_lengths)/sum(lengths):.1%} of tokens")

## 5. LoRA

In [ ]:
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16
print("bf16:", use_bf16, "| fp16:", use_fp16)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16 if use_bf16 else torch.float16,
    device_map="auto",
)
model.config.use_cache = False

lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

## 6. Train (completion-only loss)

In [ ]:
from dataclasses import dataclass
from datasets import Dataset
from transformers import Trainer, TrainingArguments

@dataclass
class CompletionCollator:
    pad_id: int

    def __call__(self, features):
        width = max(len(f["input_ids"]) for f in features)
        input_ids = torch.full((len(features), width), self.pad_id, dtype=torch.long)
        labels = torch.full((len(features), width), -100, dtype=torch.long)
        attention = torch.zeros((len(features), width), dtype=torch.long)
        for i, feature in enumerate(features):
            n = len(feature["input_ids"])
            input_ids[i, :n] = torch.tensor(feature["input_ids"], dtype=torch.long)
            labels[i, :n] = torch.tensor(feature["labels"], dtype=torch.long)
            attention[i, :n] = 1
        return {"input_ids": input_ids, "labels": labels, "attention_mask": attention}


dataset = Dataset.from_list([
    {"input_ids": x["input_ids"], "labels": x["labels"]} for x in encoded
])

args = TrainingArguments(
    output_dir=str(OUT / "checkpoints"),
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    num_train_epochs=EPOCHS,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_strategy="no",
    report_to="none",
    fp16=use_fp16,
    bf16=use_bf16,
    optim="adamw_torch",
    remove_unused_columns=False,
    dataloader_pin_memory=False,
)
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=dataset,
    data_collator=CompletionCollator(pad_id=tokenizer.pad_token_id),
    tokenizer=tokenizer,
)
trainer.train()

## 7. Save, merge, and write the merged model

In [ ]:
adapter_dir = OUT / "adapter"
merged_dir = OUT / "merged"
model.save_pretrained(adapter_dir)
merged = model.merge_and_unload()
merged.save_pretrained(merged_dir)
tokenizer.save_pretrained(merged_dir)
model = merged
print("adapter:", adapter_dir)
print("merged:", merged_dir)

## 8. Convert to GGUF Q8_0

In [ ]:
%cd /content
!rm -rf llama.cpp
!git clone --depth 1 https://github.com/ggml-org/llama.cpp
# Conversion needs gguf + sentencepiece, not llama.cpp's training requirements
# (installing those can replace Colab's CUDA torch build).
!pip install -q gguf sentencepiece protobuf

gguf_dir = OUT / "gguf"
gguf_dir.mkdir(parents=True, exist_ok=True)
gguf_path = gguf_dir / f"{MODEL_NAME}-q8_0.gguf"
!python llama.cpp/convert_hf_to_gguf.py {merged_dir} --outfile {gguf_path} --outtype q8_0
print(gguf_path, round(gguf_path.stat().st_size / 1e6, 1), "MB")

## 9. Modelfile — the chat template Ollama serves

The `TEMPLATE` block is **copied from the installed base model**, not reconstructed:

```bash
ollama show gemma3:270m --modelfile
```

If Ollama updates the base template, re-run that command and paste the new block. The
`PARAMETER` lines mirror the base model (`top_p 0.95`, `top_k 64`, stop token) plus the
sampling this app uses (`temperature 0.2`, `num_ctx 8192`).

In [ ]:
OLLAMA_TEMPLATE = r'''{{- $systemPromptAdded := false }}
{{- range $i, $_ := .Messages }}
{{- $last := eq (len (slice $.Messages $i)) 1 }}
{{- if eq .Role "user" }}<start_of_turn>user
{{- if (and (not $systemPromptAdded) $.System) }}
{{- $systemPromptAdded = true }}
{{ $.System }}
{{ end }}
{{ .Content }}<end_of_turn>
{{ if $last }}<start_of_turn>model
{{ end }}
{{- else if eq .Role "assistant" }}<start_of_turn>model
{{ .Content }}{{ if not $last }}<end_of_turn>
{{ end }}
{{- end }}
{{- end }}'''

MODELFILE = f'''FROM ./{gguf_path.name}

TEMPLATE """{OLLAMA_TEMPLATE}"""

PARAMETER temperature 0.2
PARAMETER top_p 0.95
PARAMETER top_k 64
PARAMETER num_ctx 8192
PARAMETER stop <end_of_turn>
'''

modelfile_path = gguf_dir / "Modelfile"
modelfile_path.write_text(MODELFILE, encoding="utf-8")
print(MODELFILE)

In [ ]:
print("On the machine that has Ollama:")
print()
print(f"  cd {gguf_dir}")
print(f"  ollama create {MODEL_NAME}:q8 -f Modelfile")
print(f"  ollama run {MODEL_NAME}:q8")
print()
print("Then, from the repo:")
print()
print(f"  node scripts/tiny-eval.mjs --model {MODEL_NAME}:q8 --strict")
print()
print("Route just the layer you trained, e.g. for a tune-only adapter:")
print()
print(f"  OLLAMA_MODEL_TUNE={MODEL_NAME}:q8 npm run dev")

## 10. Smoke test the merged model (optional)

Three held-out prompts (one per task when all three are selected). This is a sanity
check that the adapter produces JSON in the right shape; the real before/after numbers
come from `scripts/tiny-eval.mjs --strict` on the imported GGUF.

In [ ]:
import torch

model.eval()
picks = []
for task in sorted({r["task"] for r in eval_records}):
    picks.extend([r for r in eval_records if r["task"] == task][:1])

for record in picks:
    system, user = split_prompt(record["prompt"])
    prompt_text = render_messages(system, user)
    inputs = tokenizer(prompt_text, return_tensors="pt", add_special_tokens=False).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    generated = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False)
    print("=" * 70)
    print("task:", record["task"], "| group:", record["group"])
    print("--- generated ---")
    print(generated[:800])
    print("--- target ---")
    print(json.dumps(record["target"], ensure_ascii=False)[:800])

## 11. Notes

- **One adapter or three?** A single adapter on all three calls is the smallest
  deployment. Training per layer (`TASKS = ("tune",)`, rerun) gives three imports and
  uses the per-layer env vars in `.env.example`:
  `OLLAMA_MODEL_TUNE`, `OLLAMA_MODEL_COPY`, `OLLAMA_MODEL_FALLBACK`.
- **Why completion-only?** The prompts are ~800–1,100 tokens and the answers are
  ~50–200; training the prompt tokens would add loss on text the model already sees and
  teach it to reproduce the brief.
- **Why Q8_0?** It is the closest GGUF quantisation to the merged fp16 weights and
  still fits this 2-core CPU box; conversion is lossless up to quantisation.
- **No cloud at inference.** The GGUF is imported with `ollama create`; the app keeps
  calling `127.0.0.1:11434`. Nothing in this notebook is used at runtime.
- **Gemma license.** The base model is under the Gemma Terms of Use; the adapter and
  merged weights inherit them.
- **Data provenance.** `data/template-copy.*.jsonl` is synthetic and reproducible with
  `node scripts/make-dataset.mjs` (seeded, no model calls). The eval split contains
  held-out occasions (`data/template-copy.stats.json` → `heldOut`), so eval is not a
  paraphrase of train.